In [ ]:
from pyspark.sql import SparkSession, functions as F

spark = (SparkSession.builder
    .appName("twibot")
    .config("spark.jars.packages",
            "graphframes:graphframes:0.8.3-spark3.5-s_2.12,"
            "org.mongodb.spark:mongo-spark-connector_2.12:10.4.0")
    .config("spark.mongodb.write.connection.uri", "mongodb://mongo:27017")
    .config("spark.mongodb.read.connection.uri", "mongodb://mongo:27017")
    .config("spark.driver.memory", "3g")
    .config("spark.sql.shuffle.partitions", "32")
    .getOrCreate())
spark.sparkContext.setCheckpointDir("/tmp/ckpt")
out = "/home/jovyan/work/parquet"
print(spark.version)

In [5]:
from pyspark.sql import functions as F
out = "/home/jovyan/work/parquet"
lp    = spark.read.parquet(f"{out}/communities")
stats = spark.read.parquet(f"{out}/community_stats")
v = lp.select("id")

In [ ]:
pip install ijson

In [ ]:
import ijson

keep = set(r["id"] for r in v.select("id").collect())
rows = []
with open("/home/jovyan/work/data/twibot_dataset/user.json", "rb") as fh:
    for u in ijson.items(fh, "item"):
        if u["id"] in keep:
            pm = u.get("public_metrics") or {}
            rows.append((u["id"], u.get("created_at"), u.get("description") or "",
                         pm.get("followers_count"), pm.get("following_count"),
                         pm.get("tweet_count"), pm.get("listed_count"),
                         bool(u.get("verified"))))
print("matched users:", len(rows))

schema = ("id string, created_at string, description string, followers long, "
          "following long, tweets long, listed long, verified boolean")
uf = spark.createDataFrame(rows, schema)
uf.write.mode("overwrite").parquet(f"{out}/user_features")
print("user_features saved")

In [ ]:
from pyspark.sql import Window

uf = spark.read.parquet(f"{out}/user_features")
uf = (uf.withColumn("created", F.to_timestamp("created_at"))
        .withColumn("ym", F.date_format("created", "yyyy-MM"))
        .withColumn("no_desc", (F.length(F.trim("description")) == 0).cast("int"))
        .withColumn("ratio", F.col("following") / (F.col("followers") + 1)))

print("unparsed creation dates:", uf.filter("created IS NULL").count())

j = uf.join(lp.select("id", F.col("label").alias("community")), "id")


ym = j.groupBy("community", "ym").count()
w = Window.partitionBy("community").orderBy(F.desc("count"))
burst = (ym.withColumn("rn", F.row_number().over(w)).filter("rn = 1")
           .select("community", F.col("count").alias("top_month_n")))

agg = j.groupBy("community").agg(
    F.count("*").alias("n"),
    F.round(F.avg("no_desc"), 3).alias("no_desc_share"),
    F.round(F.expr("percentile_approx(ratio, 0.5)"), 2).alias("median_ratio"),
    F.expr("percentile_approx(tweets, 0.5)").alias("median_tweets"),
    F.round(F.avg(F.col("verified").cast("int")), 3).alias("verified_share"))

feat = (agg.join(burst, "community")
           .withColumn("burst_share", F.round(F.col("top_month_n") / F.col("n"), 3)))

full = stats.join(feat, stats.label == feat.community).drop("community", "n", "top_month_n")
full.write.mode("overwrite").parquet(f"{out}/community_features")

print("overall no_desc share:", round(uf.agg(F.avg("no_desc")).first()[0], 3))
top_ym = uf.groupBy("ym").count().agg(F.max("count")).first()[0]
print("overall busiest-month share:", round(top_ym / uf.count(), 3))

(full.filter("size >= 50 AND is_connected")
     .select("label", "size", "lift", "density", "burst_share",
             "no_desc_share", "median_ratio", "median_tweets")
     .orderBy(F.desc("lift")).show(15))

In [ ]:
cf = spark.read.parquet(f"{out}/community_features")
elig = cf.filter("size >= 50 AND is_connected")

p = elig.select("label", "size", "bot_ratio", "lift", "density",
                "burst_share", "no_desc_share", "median_ratio").toPandas()

p["median_ratio"] = p["median_ratio"].clip(upper=10)
for c in ["density", "burst_share", "no_desc_share", "median_ratio"]:
    p[c + "_r"] = p[c].rank(pct=True)

p["coord_score"] = (0.30 * p["burst_share_r"] + 0.30 * p["no_desc_share_r"]
                    + 0.20 * p["density_r"] + 0.20 * p["median_ratio_r"]).round(3)
p = p.sort_values("coord_score", ascending=False).reset_index(drop=True)

print(p[["label", "size", "coord_score", "lift", "bot_ratio"]].head(10))


print("\nSpearman(score, lift):", round(p["coord_score"].corr(p["lift"], method="spearman"), 3))
print("mean lift, top 10 by score:", round(p.head(10)["lift"].mean(), 2),
      "| all eligible:", round(p["lift"].mean(), 2))

keep_cols = ["label", "size", "bot_ratio", "lift", "density", "burst_share",
             "no_desc_share", "median_ratio", "coord_score"]
spark.createDataFrame(p[keep_cols]).write.mode("overwrite").parquet(f"{out}/community_scores")
print("community_scores saved")